# 01. Merge and derive analysis variables

This notebook combines the 2021–2024 Student Health Examination datasets and creates the variables required for analysis.

## Main tasks

- Harmonize and merge the annual datasets.
- Clean and derive the analysis variables.
- Define metabolic syndrome using IDF (`MS`, primary) and modified ATP III (`MS_atp3`, sensitivity) criteria.
- Apply age-specific waist circumference cut-offs and preserve missing components as indeterminate.
- Record the physical activity question used for each school level.
- Save the processed dataset, participant flow, component summary, and definition agreement table.

## Expected counts

The processed dataset should contain **13,984 participants**: 1,586 elementary, 5,484 middle, and 6,914 high-school students. The notebook stops if these counts do not match.


In [1]:
# Imports and paths
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Opt into the pandas behavior where replace does not silently change dtype;
# the conversion is then requested explicitly with infer_objects.
pd.set_option("future.no_silent_downcasting", True)

# Use environment-based paths before publication; local paths are retained here.
BASE = Path(os.environ.get("METS_DATA", r"D:\eda\mets\data"))
OUT  = Path(os.environ.get("METS_OUT",  r"D:\eda\mets\output"))
for _d in (BASE, BASE/"raw", BASE/"work", OUT, OUT/"tables", OUT/"figures", OUT/"logs"):
    _d.mkdir(parents=True, exist_ok=True)

RAW = {2021: BASE/"raw"/"2021.csv", 2022: BASE/"raw"/"2022.csv",
       2023: BASE/"raw"/"2023.csv", 2024: BASE/"raw"/"2024.csv"}

# The 2024 source file uses CP949; earlier files use the default encoding.
ENC = {2021: None, 2022: None, 2023: None, 2024: "cp949"}

GROWTH   = BASE/"raw"/"Growth_table_BMI.csv"
OUT_CSV  = BASE/"work"/"Obesity_Data_proprecessing.csv"
# Each notebook writes its own segment of the participant flow. Notebook 04
# joins the segments in order, so re-running one notebook cannot leave
# duplicated steps behind in a shared file.
OUT_FLOW = OUT/"tables"/"flow_01.csv"

# Validate all required inputs before processing.
_expected = list(RAW.values()) + [GROWTH]
_absent = [f for f in _expected if not f.exists()]
if _absent:
    raise FileNotFoundError(
        "These data files were not found:\n"
        + "".join("    %s\n" % f for f in _absent)
        + "\nLooked in : %s\n" % BASE.resolve()
        + "Running in: %s\n" % Path.cwd()
        + "\nThe default location is a 'data' folder beside 'notebooks'. If the\n"
          "data is somewhere else, set METS_DATA before running this cell:\n"
          "    import os\n"
          '    os.environ["METS_DATA"] = r"D:\\path\\to\\data"\n'
          '    os.environ["METS_OUT"]  = r"D:\\path\\to\\output"\n'
          "\nThe five files belong in <METS_DATA>/raw/ :\n"
          "    2021.csv  2022.csv  2023.csv  2024.csv  Growth_table_BMI.csv"
    )
print("data   :", BASE.resolve())
print("output :", OUT.resolve())

# Analysis constants
SEX_MALE = 1
SEX_FEMALE = 0

MIN_AGE = 10
ADULT_AGE = 16

BP_SYSTOLIC_CUTOFF = 130
BP_DIASTOLIC_CUTOFF = 85

TG_CUTOFF = 150
GLUCOSE_CUTOFF = 100

HDL_CUTOFF_ADULT_MALE = 40      # ATP III, and IDF from 16 years of age
HDL_CUTOFF_ADULT_FEMALE = 50
HDL_CUTOFF_PEDIATRIC = 40      # IDF below 16 years; the same for both sexes

# Adult waist cut-offs: IDF is primary; KSSO is available for comparison.
WAIST_ADULT_SOURCE = "idf"
WAIST_ADULT_CUTOFF = {
    "idf":  {SEX_MALE: 90.0, SEX_FEMALE: 80.0},
    "ksso": {SEX_MALE: 90.0, SEX_FEMALE: 85.0},
}

MS_COLS = ["waist", "sysBP", "diaBP", "HDL", "TG", "glucose"]
# Participant-flow helpers

FLOW = []

def flow_start(d, stage):
    """Begin a new participant flow record."""
    FLOW.clear()
    FLOW.append({"step": 0, "stage": stage, "excluded": 0, "remaining": len(d)})
    print(f"{'':2}{stage:<55} n = {len(d):>9,}")
    return d

def keep(d, mask, stage):
    """Keep the rows where mask is True and record how many were excluded."""
    mask = mask.reindex(d.index).fillna(False).astype(bool)
    n_before = len(d)
    out = d.loc[mask].copy()
    FLOW.append({"step": len(FLOW), "stage": stage,
                 "excluded": n_before - len(out), "remaining": len(out)})
    print(f"  - {stage:<53} -{n_before-len(out):>8,}  ->  {len(out):>9,}")
    return out


data   : D:\eda\mets\data
output : D:\eda\mets\output


## 1. Merge

In [2]:
df_2021 = pd.read_csv(RAW[2021], low_memory=False)
df_2022 = pd.read_csv(RAW[2022], low_memory=False)
df_2023 = pd.read_csv(RAW[2023], low_memory=False)
df_2024 = pd.read_csv(RAW[2024], low_memory=False, encoding=ENC[2024])

for y, d in zip([2021, 2022, 2023, 2024], [df_2021, df_2022, df_2023, df_2024]):
    print(f"{y}: {len(d):>8,} rows")

# C1 (Editor 3): the flow diagram quotes these, so write them to a file.
year_counts = pd.DataFrame(
    [{"year": y, "records": len(d)} for y, d in
     zip([2021, 2022, 2023, 2024], [df_2021, df_2022, df_2023, df_2024])])
year_counts.to_csv(OUT/"tables"/"records_by_year.csv", index=False,
                   encoding="utf-8-sig")
print(f"total {int(year_counts.records.sum()):,}"
      f"  -> saved tables/records_by_year.csv")


2021:   97,787 rows
2022:   92,722 rows
2023:   87,183 rows
2024:   89,211 rows
total 366,903  -> saved tables/records_by_year.csv


In [3]:
def rename_select(df, column_map):
    """Rename the source columns to the analysis names and keep only those."""
    missing = [source for source in column_map if source not in df.columns]
    if missing:
        raise KeyError("columns not present in the source file: "
                       + ", ".join(missing))
    df = df.rename(columns=column_map)
    return df[list(column_map.values())]


In [4]:
# 2021 source-to-analysis column map
col = {
    '자아신체상(체형)': 'body_image',

    '공학여부': 'school_type',
    '시도': 'city',
    '학교급': 'school_level',
    '학년': 'grade',
    '성별': 'sex',

    '생년월일': 'birth',
    '건강검진일': 'exam_date',

    '키': 'height',
    '몸무게': 'weight',
    '허리둘레': 'waist',
    '수축기': 'sysBP',
    '이완기': 'diaBP',

    '혈당(식전)(mg/dl)': 'glucose',
    '총콜레스테롤(mg/dl)': 'chol',
    'hdl(mg/dl)': 'HDL',
    '중성지방(mg/dl)': 'TG',
    'ldl(mg/dl)': 'LDL',
    'AST(U/L)': 'AST',
    'ALT(U/L)': 'ALT',

    '라면': 'DB_1',
    '음료수': 'DB_2',
    '패스트푸드': 'DB_3',
    '육류': 'DB_4',
    '우유유제품': 'DB_5',
    '과일': 'DB_6',
    '채소(김치제외)': 'DB_7',
    '아침식사': 'DB_8',

    '다이어트경험_답변1': 'DB_9_1',
    '다이어트경험_답변2': 'DB_9_2',
    '다이어트경험_답변3': 'DB_9_3',
    '다이어트경험_답변4': 'DB_9_4',

    '주3회이상운동': 'PA_1st',
    '하루30분이상운동': 'PA_2nd',

    '하루수면량': 'sleep',

    '하루2시간이상게임_초': 'internet_1st',
    '하루2시간이상게임_중고': 'internet_2nd',

    '괴롭힘따돌림_초': 'bully_1st',
    '괴롭힘따돌림_중고': 'bully_2nd',
    '가정문제걱정': 'family_worry',

    '가출생각_초': 'runaway_1st',
    '가출생각_중고': 'runaway_2nd',
    '상담희망_초': 'want_counseling_1st',
    '상담희망_중고': 'want_counseling_2nd',
    '상담요청_초': 'request_counseling_1st',
    '상담요청_중고': 'request_counseling_2nd',
}

df_2021 = rename_select(df_2021, col)


In [5]:
# 2022 source-to-analysis column map
col = {
    '자아신체상체형': 'body_image',

    '공학여부': 'school_type',
    '시도': 'city',
    '학교급': 'school_level',
    '학년': 'grade',
    '성별': 'sex',

    '생년월일': 'birth',
    '건강검진일': 'exam_date',

    '키': 'height',
    '몸무게': 'weight',
    '허리둘레': 'waist',
    '수축기': 'sysBP',
    '이완기': 'diaBP',

    '혈당식전mgdl': 'glucose',
    '총콜레스테롤mgdl': 'chol',
    'hdlmgdl': 'HDL',
    '중성지방mgdl': 'TG',
    'ldlmgdl': 'LDL',
    'astul': 'AST',
    'altul': 'ALT',

    '라면': 'DB_1',
    '음료수': 'DB_2',
    '패스트푸드': 'DB_3',
    '육류': 'DB_4',
    '우유유제품': 'DB_5',
    '과일': 'DB_6',
    '채소김치제외': 'DB_7',
    '아침식사': 'DB_8',

    '다이어트경험_답변1': 'DB_9_1',
    '다이어트경험_답변2': 'DB_9_2',
    '다이어트경험_답변3': 'DB_9_3',
    '다이어트경험_답변4': 'DB_9_4',

    '주3회이상운동': 'PA_1st',
    '하루30분이상운동': 'PA_2nd',

    '하루수면량': 'sleep',

    '하루2시간이상게임_초': 'internet_1st',
    '하루2시간이상게임_중고': 'internet_2nd',

    '괴롭힘따돌림_초': 'bully_1st',
    '괴롭힘따돌림_중고': 'bully_2nd',
    '가정문제걱정': 'family_worry',

    '가출생각_초': 'runaway_1st',
    '가출생각중고': 'runaway_2nd',
    '학교문제상담희망_초': 'want_counseling_1st',
    '학교문제상담희망_중고': 'want_counseling_2nd',
    '상담요청_초': 'request_counseling_1st',
    '상담요청_중고': 'request_counseling_2nd',
}

df_2022 = rename_select(df_2022, col)


In [6]:
# 2023 source-to-analysis column map
col = {
    '자아신체상체형': 'body_image',

    '공학여부': 'school_type',
    '시도': 'city',
    '학교급': 'school_level',
    '학년': 'grade',
    '성별': 'sex',

    '생년월일': 'birth',
    '건강검진일': 'exam_date',

    '키': 'height',
    '몸무게': 'weight',
    '허리둘레': 'waist',
    '수축기혈압': 'sysBP',
    '이완기혈압': 'diaBP',

    '식전혈당': 'glucose',
    '총콜레스테롤': 'chol',
    '고밀도지단백콜레스테롤_HDL': 'HDL',
    '중성지방_TG': 'TG',
    '저밀도지단백콜레스테롤_LDL': 'LDL',
    '간세포효소_AST': 'AST',
    '간세포효소_ALT': 'ALT',

    '라면': 'DB_1',
    '음료수': 'DB_2',
    '패스트푸드': 'DB_3',
    '육류': 'DB_4',
    '우유유제품': 'DB_5',
    '과일': 'DB_6',
    '채소김치제외': 'DB_7',
    '아침식사': 'DB_8',

    '다이어트경험_답변1': 'DB_9_1',
    '다이어트경험_답변2': 'DB_9_2',
    '다이어트경험_답변3': 'DB_9_3',
    '다이어트경험_답변4': 'DB_9_4',

    '주3회이상운동': 'PA_1st',
    '하루30분이상운동': 'PA_2nd',

    '하루수면량': 'sleep',

    '하루2시간이상게임_초': 'internet_1st',
    '하루2시간이상게임_중고': 'internet_2nd',

    '괴롭힘따돌림': 'bully',
    '가정문제걱정': 'family_worry',

    '가출생각': 'runaway',
    '학교문제상담희망_초': 'want_counseling_1st',
    '학교문제상담희망_중고': 'want_counseling_2nd',
    '상담요청': 'request_counseling',
}

df_2023 = rename_select(df_2023, col)


In [7]:
# 2024 source-to-analysis column map
col = {
    '자아신체상체형': 'body_image',

    '공학여부': 'school_type',
    '시도': 'city',
    '학교급': 'school_level',
    '학년': 'grade',
    '성별': 'sex',

    '생년월일': 'birth',
    '건강검진일': 'exam_date',

    '키': 'height',
    '몸무게': 'weight',
    '허리둘레': 'waist',
    '수축기혈압': 'sysBP',
    '이완기혈압': 'diaBP',

    '식전혈당': 'glucose',
    '총콜레스테롤': 'chol',
    '고밀도지단백콜레스테롤(HDL)': 'HDL',
    '중성지방(TG)': 'TG',
    '저밀도지단백콜레스테롤(LDL)': 'LDL',
    '간세포효소(AST)': 'AST',
    '간세포효소(ALT)': 'ALT',

    '라면': 'DB_1',
    '음료수': 'DB_2',
    '패스트푸드': 'DB_3',
    '육류': 'DB_4',
    '우유유제품': 'DB_5',
    '과일': 'DB_6',
    '채소김치제외': 'DB_7',
    '아침식사': 'DB_8',

    '다이어트경험_답변1': 'DB_9_1',
    '다이어트경험_답변2': 'DB_9_2',
    '다이어트경험_답변3': 'DB_9_3',
    '다이어트경험_답변4': 'DB_9_4',

    '주3회이상운동': 'PA_1st',
    '하루30분이상운동': 'PA_2nd',

    '하루수면량': 'sleep',

    '2시간이상게임': 'internet',

    '괴롭힘따돌림': 'bully',
    '가정문제걱정': 'family_worry',

    '가출생각': 'runaway',
    '상담희망': 'want_counseling',
    '고민상담요청': 'request_counseling',
}

df_2024 = rename_select(df_2024, col)


In [8]:
# Merge annual files and initialise participant-flow tracking.
df_2021['year'], df_2022['year'], df_2023['year'], df_2024['year'] = 2021, 2022, 2023, 2024
df_all = pd.concat([df_2021, df_2022, df_2023, df_2024], axis=0, ignore_index=True)

print("=== Participant flow ===")
df_all = flow_start(df_all, "Merged Student Health Examination records, 2021-2024")


=== Participant flow ===
  Merged Student Health Examination records, 2021-2024    n =   366,903


## 2. Data cleaning

In [9]:
# Parse dates after removing separators and trailing decimal notation.
for col in ['birth', 'exam_date']:
    s = df_all[col].astype(str).str.replace(r'\.0|-', '', regex=True)
    df_all[col] = pd.to_datetime(s, format='%Y%m%d', errors='coerce')

# Convert analysis fields to numeric; invalid text becomes missing.
target_cols = ['body_image', 'height', 'weight', 'waist', 'sysBP', 'diaBP', 'glucose',
               'chol', 'HDL', 'TG', 'LDL', 'AST', 'ALT', 'sleep', 'family_worry'] + \
              [f'DB_{i}' for i in range(1, 9)] + [f'DB_9_{i}' for i in range(1, 5)] + \
              ['PA_1st', 'PA_2nd', 'internet', 'internet_1st', 'internet_2nd',
               'bully', 'bully_1st', 'bully_2nd',
               'runaway', 'runaway_1st', 'runaway_2nd',
               'want_counseling', 'want_counseling_1st', 'want_counseling_2nd',
               'request_counseling', 'request_counseling_1st', 'request_counseling_2nd']

for col in target_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

# Canonical interpretation used throughout the analysis:
#   want_counseling    = need for counseling about school or home problems
#   request_counseling = need for counseling about personal worries
# The raw Korean item names vary by year; the source-to-analysis maps above
# preserve the item identity before school-level versions are harmonized.

# Combine equivalent school-level questionnaire fields.
for name in ['internet', 'bully', 'runaway', 'want_counseling', 'request_counseling']:
    df_all[name] = (df_all[name]
                    .fillna(df_all[f'{name}_1st'])
                    .fillna(df_all[f'{name}_2nd']))

# Physical activity questions differ between elementary and secondary schools.
df_all['PA'] = np.nan
# The school-level variable still contains the original Korean labels.
is_elementary = df_all['school_level'].astype(str).str.strip().eq('초')

df_all.loc[is_elementary, 'PA'] = df_all.loc[is_elementary, 'PA_1st'].map({1: 1, 2: 0})

df_all.loc[~is_elementary, 'PA'] = df_all.loc[~is_elementary, 'PA_2nd'].map(
    {1: 0, 2: 0, 3: 1, 4: 1})

# Retain the question source for reporting.
df_all['PA_source'] = np.where(
    is_elementary,
    'elementary: vigorous exercise >=3 days/week',
    'middle/high: vigorous exercise 30 min-1 h/day on >=3 days/week')

# Remove the source-specific fields after harmonization.
df_all.drop(columns=[c for c in df_all.columns if '_1st' in c or '_2nd' in c],
            inplace=True)


In [10]:
# The source file labels sex and school level in Korean, so the labels are
# recoded here: sex 'M' -> 1 and 'F' -> 2, school level elementary -> 1,
# middle -> 2 and high -> 3. Sex stays 1/2 until the growth-chart merge
# and is recoded to 1/0 after it.
df_all['sex'] = (df_all['sex'].replace({'남': 1, '여': 2})
                 .infer_objects(copy=False))
df_all['school_level'] = (df_all['school_level'].astype(str).str.strip()
                          .replace({'초': 1, '중': 2, '고': 3})
                          .infer_objects(copy=False))

df = df_all.copy()


In [11]:
df = keep(df, df[['height', 'weight']].notna().all(axis=1),
          "Missing height or weight")


  - Missing height or weight                              -       1  ->    366,902


In [12]:
# Derive age at examination and a grade index spanning grades 1–12.
df['age'] = (df['exam_date'] - df['birth']).dt.days / 365.25

is_elementary = df['school_level'] == 1
is_middle = df['school_level'] == 2
is_high = df['school_level'] == 3
df['global_grade'] = np.nan
df.loc[is_elementary, 'global_grade'] = df.loc[is_elementary, 'grade']
df.loc[is_middle, 'global_grade'] = df.loc[is_middle, 'grade'] + 6
df.loc[is_high, 'global_grade'] = df.loc[is_high, 'grade'] + 9

# Exclude invalid dates and large age–grade inconsistencies.
age_gap = (df['year'] - df['birth'].dt.year) - (df['global_grade'] + 6)

df = keep(df, (df['exam_date'] > df['birth']) & (age_gap.abs() < 3) & (df['age'] > 5),
          "Invalid date, grade-age mismatch, or age <= 5")


  - Invalid date, grade-age mismatch, or age <= 5         -     165  ->    366,737


In [13]:
# Restrict the cohort to ages covered by the MetS definition.
df = keep(df, df['age'] >= MIN_AGE, f"Age < {MIN_AGE} years")

print("\n[Missing values among records aged %d or older]" % MIN_AGE)
print(df[MS_COLS].isna().sum().to_string())
print()

# Requiring all components identifies students who received the biochemical exam.
df = keep(df, df[MS_COLS].notna().all(axis=1),
          "MetS components not measured (no biochemical exam)")


  - Age < 10 years                                        -  85,064  ->    281,673

[Missing values among records aged 10 or older]
waist      266122
sysBP      194254
diaBP      194256
HDL        266079
TG         266014
glucose    265918

  - MetS components not measured (no biochemical exam)    - 266,674  ->     14,999


In [14]:
# Exclude contradictory or unanswered weight-control checkboxes.
conflicting = (df['DB_9_1'].notnull()
               & df[['DB_9_2', 'DB_9_3', 'DB_9_4']].notnull().any(axis=1))
unanswered = df[['DB_9_1', 'DB_9_2', 'DB_9_3', 'DB_9_4']].isnull().all(axis=1)

df = keep(df, ~conflicting & ~unanswered,
          "Weight-control items: conflicting or unanswered")


  - Weight-control items: conflicting or unanswered       -   1,009  ->     13,990


In [15]:
# Validate the eight dietary responses.
dietary_items = [f'DB_{i}' for i in range(1, 9)]
df = keep(df, df[dietary_items].isin([1, 2, 3, 4]).all(axis=1),
          "Dietary items invalid or missing")

# Derive reported weight-control behaviors from checked items.
df['weight_control_diet']       = df['DB_9_2'].notnull().astype(int)
df['weight_control_medication'] = df['DB_9_3'].notnull().astype(int)
df['weight_control_exercise']   = df['DB_9_4'].notnull().astype(int)
df['weight_control_attempt']    = df[['weight_control_diet',
                                      'weight_control_exercise',
                                      'weight_control_medication']].any(axis=1).astype(int)

df.drop(columns=['DB_9_1', 'DB_9_2', 'DB_9_3', 'DB_9_4'], inplace=True)


  - Dietary items invalid or missing                      -       6  ->     13,984


In [16]:
# Harmonize school-type labels across years.
school_map = {
    '1. 공학': 'coeducational', '남녀공학': 'coeducational', '남여공학': 'coeducational',
    '3. 여자': 'girls', '여자': 'girls', '녀': 'girls',
    '2. 남자': 'boys', '남자': 'boys', '남': 'boys',
}
df['school_type'] = df['school_type'].astype(str).str.strip().map(school_map)

print(df['school_type'].value_counts(dropna=False).to_string())


school_type
coeducational    9897
boys             2453
girls            1634


In [17]:
# EatingIndex ranges from 0 to 24; higher scores indicate a healthier diet.
rename_map = {"DB_1": "ramen", "DB_2": "drink", "DB_3": "fastfood", "DB_4": "meat",
              "DB_5": "dairy", "DB_6": "fruit", "DB_7": "vegetable", "DB_8": "breakfast"}
df = df.rename(columns=rename_map)

# Score frequent recommended foods positively and limited foods in reverse.
score_more_is_better = {1: 0, 2: 1, 3: 2, 4: 3}
score_less_is_better = {1: 3, 2: 2, 3: 1, 4: 0}

recommended = ["meat", "dairy", "fruit", "vegetable", "breakfast"]
limited = ["ramen", "drink", "fastfood"]

for c in recommended:
    df[f"EI_{c}"] = df[c].map(score_more_is_better)
for c in limited:
    df[f"EI_{c}"] = df[c].map(score_less_is_better)

# Require all eight items when calculating the total.
ei_cols = [f"EI_{c}" for c in (recommended + limited)]
df["EatingIndex"] = df[ei_cols].sum(axis=1, min_count=len(ei_cols))

print("EatingIndex missing:", df["EatingIndex"].isna().sum())
print(df["EatingIndex"].describe().round(2).to_string())


EatingIndex missing: 0
count    13984.00
mean        13.82
std          2.48
min          4.00
25%         12.00
50%         14.00
75%         15.00
max         24.00


In [18]:
# Calculate BMI z-scores using the LMS method and 2017 Korean growth charts.
df['BMI_calc'] = df['weight'] / ((df['height'] / 100) ** 2)
growth = pd.read_csv(GROWTH)[['sex', 'months', 'L', 'M', 'S']].dropna()

df['months'] = (df['age'] * 12).round().astype(int).clip(upper=growth['months'].max())
df = pd.merge(df, growth, on=['sex', 'months'], how='left')
df['bmi_z'] = ((df['BMI_calc'] / df['M']) ** df['L'] - 1) / (df['L'] * df['S'])

def classify_obesity(z):
    """Korean growth chart classification: the 95th, 85th and 5th percentiles."""
    if pd.isna(z):
        return np.nan
    if z >= 1.645:
        return 'Obesity'
    if z >= 1.036:
        return 'Overweight'
    if z <= -1.645:
        return 'Underweight'
    return 'Normal'

df['obesity_class'] = df['bmi_z'].apply(classify_obesity)

# Report the obesity profile of the screened cohort.
print("=== Obesity classification ===")
for label, count in df['obesity_class'].value_counts(dropna=False).items():
    print(f"  {str(label):<12}{count:>7,}  ({count/len(df)*100:5.2f}%)")
print(f"  BMI z-score: mean {df['bmi_z'].mean():.2f} (SD {df['bmi_z'].std():.2f})")
print(f"  bmi_z >= 1.645 : {(df['bmi_z'] >= 1.645).mean()*100:.1f}%")


=== Obesity classification ===
  Obesity      13,123  (93.84%)
  Overweight      754  ( 5.39%)
  Normal          100  ( 0.72%)
  Underweight       7  ( 0.05%)
  BMI z-score: mean 2.48 (SD 0.80)
  bmi_z >= 1.645 : 93.8%


In [19]:
# Remove identifying dates after age derivation.
df = df.drop(columns=['birth', 'exam_date'])


In [20]:
# Final coding: sex 1/0 and binary questionnaire items 1/0.
df['sex'] = df['sex'].replace({2: SEX_FEMALE})
for c in ['runaway', 'bully', 'request_counseling', 'want_counseling',
          'internet', 'family_worry']:
    df[c] = df[c].replace({2: 0})

print(df['sex'].value_counts().rename({SEX_MALE: 'Male(1)',
                                       SEX_FEMALE: 'Female(0)'}).to_string())


sex
Male(1)      8610
Female(0)    5374


In [21]:
# Waist cut-off: 2007 Korean age/sex p90 below 16; IDF adult threshold from 16.
# The 2017 charts used above for BMI z-scores carry no waist percentiles.
df['waist'] = pd.to_numeric(df['waist'], errors='coerce')
df['age_floor'] = np.floor(df['age']).astype('Int64')

# Sex- and age-specific 90th percentiles (cm), ages 10-15.
wc_p90_map = pd.DataFrame({
    "sex":       [SEX_MALE] * 6 + [SEX_FEMALE] * 6,
    "age_floor": [10, 11, 12, 13, 14, 15] * 2,
    "wc_p90":    [79.3, 82.2, 85.2, 87.1, 89.5, 89.7,     # male
                  73.5, 76.8, 77.3, 77.8, 78.5, 78.8],    # female
})

# Drop any prior merge result so the cell can be rerun safely.
df = df.drop(columns=["wc_p90"], errors="ignore").merge(
    wc_p90_map, on=["sex", "age_floor"], how="left")
df["adult_wc_cut"] = df["sex"].map(WAIST_ADULT_CUTOFF[WAIST_ADULT_SOURCE])

# Select one waist cut-off per participant.
df["wc_cut"] = df["wc_p90"]
is_adult = df["age"] >= ADULT_AGE
df.loc[is_adult, "wc_cut"] = df.loc[is_adult, "adult_wc_cut"]

print("wc_cut missing:", int(df['wc_cut'].isna().sum()))
print(df.groupby(is_adult)['wc_cut'].agg(['min', 'max', 'count']).to_string())


wc_cut missing: 0
        min   max  count
age                     
False  73.5  89.7  11178
True   80.0  90.0   2806


In [22]:
# IDF is the primary outcome; modified ATP III is the sensitivity outcome.
# The source database does not contain medication-treatment indicators or a
# prior diabetes-diagnosis field. Classification therefore operationalizes the
# IDF criteria from the measured examination values available in this dataset.
# Missing components remain indeterminate rather than being treated as normal.
def exceeds(value, cutoff):
    """Return True at/above the cut-off, False below, or None if missing."""
    if pd.isna(value):
        return None
    return bool(value >= cutoff)

def below(value, cutoff):
    """Return True below the cut-off, False otherwise, or None if missing."""
    if pd.isna(value):
        return None
    return bool(value < cutoff)

def blood_pressure_criterion(systolic, diastolic):
    """Evaluate BP; one normal and one missing reading remains indeterminate."""
    high_systolic = exceeds(systolic, BP_SYSTOLIC_CUTOFF)
    high_diastolic = exceeds(diastolic, BP_DIASTOLIC_CUTOFF)

    if high_systolic is True or high_diastolic is True:
        return True
    if high_systolic is None or high_diastolic is None:
        return None
    return False

def hdl_cutoff(criteria, is_adult, is_male):
    """Return the applicable HDL cut-off."""
    if criteria == "ATP3":
        return HDL_CUTOFF_ADULT_MALE if is_male else HDL_CUTOFF_ADULT_FEMALE

    if is_adult:
        return HDL_CUTOFF_ADULT_MALE if is_male else HDL_CUTOFF_ADULT_FEMALE
    return HDL_CUTOFF_PEDIATRIC

def combine(criterion_flags, required):
    """Return positive, negative, or indeterminate from three-state criteria."""
    n_met = 0
    n_unknown = 0
    for flag in criterion_flags:
        if flag is True:
            n_met += 1
        elif flag is None:
            n_unknown += 1

    if n_met >= required:
        return 1
    if n_met + n_unknown < required:
        return 0
    return None

def criteria_for_one(row, criteria):
    """Evaluate all five components for one participant."""
    age = row["age"]
    sex = row["sex"]

    # Missing age is handled by classify_one().
    is_adult = (not pd.isna(age)) and age >= ADULT_AGE
    is_male = sex == SEX_MALE

    # The HDL cut-off depends on sex, so an unexpected sex code leaves the
    # criterion undecided rather than guessing a threshold.
    if sex == SEX_MALE or sex == SEX_FEMALE:
        hdl = below(row["HDL"], hdl_cutoff(criteria, is_adult, is_male))
    else:
        hdl = None

    return {
        "wc": exceeds(row["waist"], row["wc_cut"]),
        "bp": blood_pressure_criterion(row["sysBP"], row["diaBP"]),
        "hdl": hdl,
        "tg": exceeds(row["TG"], TG_CUTOFF),
        "glu": exceeds(row["glucose"], GLUCOSE_CUTOFF),
    }

def classify_one(row, criteria):
    """Return case status and component results for one participant."""
    parts = criteria_for_one(row, criteria)

    age = row["age"]
    # IDF is not applied below age 10.
    if pd.isna(age) or age < MIN_AGE:
        return None, parts

    # ATP III: any 3 of 5 criteria.
    if criteria == "ATP3":
        status = combine([parts["wc"], parts["bp"], parts["hdl"],
                          parts["tg"], parts["glu"]], required=3)
        return status, parts

    # IDF: central obesity plus at least 2 of the other 4 criteria.
    if parts["wc"] is False:
        return 0, parts
    if parts["wc"] is None:
        return None, parts

    status = combine([parts["bp"], parts["hdl"], parts["tg"], parts["glu"]],
                     required=2)
    return status, parts

def define_mets(d, criteria="IDF"):
    """Classify all participants and return components plus MetS status."""
    statuses = []
    criteria_met = []

    for _, row in d.iterrows():
        status, parts = classify_one(row, criteria)
        statuses.append(status)
        criteria_met.append(parts)

    out = pd.DataFrame(criteria_met, index=d.index)
    out["MS"] = pd.Series(statuses, index=d.index, dtype="float64")
    return out

# Create primary and sensitivity outcomes.
idf = define_mets(df, "IDF")     # primary outcome
atp = define_mets(df, "ATP3")    # sensitivity analysis

df["MS"] = idf["MS"]
df["MS_atp3"] = atp["MS"]
for c in ["wc", "bp", "hdl", "tg", "glu"]:
    df[f"comp_{c}"] = idf[c]

# Separate the IDF pediatric and adult thresholds at age 16.
df["age_band"] = "10-<16"
df.loc[df["age"] >= ADULT_AGE, "age_band"] = ">=16"

for label, result in [("IDF (primary)    ", idf),
                      ("ATP III (sensitivity)", atp)]:
    print("%-22s positive %6d (%5.2f%%) / negative %6d / indeterminate %d"
          % (label,
             int(result.MS.eq(1).sum()), result.MS.mean() * 100,
             int(result.MS.eq(0).sum()), int(result.MS.isna().sum())))

print()
print(df.groupby("age_band")[["MS", "MS_atp3"]]
        .agg(["count", "sum", "mean"]).round(4).to_string())


IDF (primary)          positive   1787 (12.78%) / negative  12197 / indeterminate 0
ATP III (sensitivity)  positive   2271 (16.24%) / negative  11713 / indeterminate 0

             MS                 MS_atp3                
          count     sum    mean   count     sum    mean
age_band                                               
10-<16    11178  1332.0  0.1192   11178  1796.0  0.1607
>=16       2806   455.0  0.1622    2806   475.0  0.1693


In [23]:
# Component prevalence, checked here where the definition is written.
# The component breakdown is built in notebook 03 on the analytic sample.
COMPONENT_LABEL = {
    "wc":  "Central obesity (WC >= cut-off)",
    "bp":  "Elevated BP (>=130/85 mmHg)",
    "hdl": "Reduced HDL cholesterol",
    "tg":  "Elevated triglycerides (>=150 mg/dL)",
    "glu": "Elevated fasting glucose (>=100 mg/dL)",
}

s1 = pd.DataFrame({
    "Component":  [COMPONENT_LABEL[c] for c in COMPONENT_LABEL],
    "n_assessed": [int(idf[c].notna().sum()) for c in COMPONENT_LABEL],
    "n_positive": [int(idf[c].eq(True).sum()) for c in COMPONENT_LABEL],
    "n_missing":  [int(idf[c].isna().sum()) for c in COMPONENT_LABEL],
})
s1["prevalence_%"] = (s1.n_positive / s1.n_assessed * 100).round(1)
print(s1.to_string(index=False))

# Agreement between the two definitions, on the same pre-exclusion records.
both = df[["MS", "MS_atp3"]].dropna()
print("\nrows = IDF (primary)  /  columns = ATP III (sensitivity)")
print(pd.crosstab(both.MS, both.MS_atp3, margins=True).to_string())

n11 = int(((both.MS == 1) & (both.MS_atp3 == 1)).sum())
n10 = int(((both.MS == 1) & (both.MS_atp3 == 0)).sum())
n01 = int(((both.MS == 0) & (both.MS_atp3 == 1)).sum())
n00 = int(((both.MS == 0) & (both.MS_atp3 == 0)).sum())
n = len(both)

observed_agreement = (n11 + n00) / n
expected_agreement = ((n11 + n10) * (n11 + n01)
                      + (n01 + n00) * (n10 + n00)) / n ** 2
kappa = (observed_agreement - expected_agreement) / (1 - expected_agreement)

print(f"\ndiscordant {n10+n01:,} ({(n10+n01)/n*100:.2f}%) | kappa {kappa:.4f}")
print(f"prevalence  IDF {both.MS.mean()*100:.2f}%  /  "
      f"ATP III {both.MS_atp3.mean()*100:.2f}%")

# Every IDF case must also meet the ATP III count rule.
assert n10 == 0, f"{n10} participants are IDF-positive but ATP III-negative"
print(">>> consistency check passed: every IDF case is also an ATP III case")

print("These counts are before the missing-value and implausible-value\n"
      "exclusions. Notebook 03 writes the component table on the 13,670\n"
      "participants the manuscript reports.")


                             Component  n_assessed  n_positive  n_missing  prevalence_%
       Central obesity (WC >= cut-off)       13984        9916          0          70.9
           Elevated BP (>=130/85 mmHg)       13984        1309          0           9.4
               Reduced HDL cholesterol       13984        2514          0          18.0
  Elevated triglycerides (>=150 mg/dL)       13984        2758          0          19.7
Elevated fasting glucose (>=100 mg/dL)       13984        3094          0          22.1

rows = IDF (primary)  /  columns = ATP III (sensitivity)
MS_atp3    0.0   1.0    All
MS                         
0.0      11713   484  12197
1.0          0  1787   1787
All      11713  2271  13984

discordant 484 (3.46%) | kappa 0.8608
prevalence  IDF 12.78%  /  ATP III 16.24%
>>> consistency check passed: every IDF case is also an ATP III case
These counts are before the missing-value and implausible-value
exclusions. Notebook 03 writes the component table on the 13

In [24]:
# Retain the three grades sampled by the national programme and recode them 1–3.
df = keep(df, df['global_grade'].isin([4, 7, 10]),
          "Grade other than elementary 4 / middle 1 / high 1")
df['global_grade'] = df['global_grade'].map({4: 1, 7: 2, 10: 3})

print("\n--- global_grade after recoding ---")
print(df['global_grade'].value_counts().sort_index().to_string())


  - Grade other than elementary 4 / middle 1 / high 1     -       0  ->     13,984

--- global_grade after recoding ---
global_grade
1    1586
2    5484
3    6914


In [25]:
# Confirm that outcome definitions did not change cohort membership.
EXPECTED_N = 13_984
EXPECTED_BY_GRADE = {1: 1586, 2: 5484, 3: 6914}

actual_by_grade = {int(k): int(v) for k, v in
                   df['global_grade'].value_counts().sort_index().items()}

print(f"rows                 : {len(df):,}  (expected {EXPECTED_N:,})")
print(f"by school level      : {actual_by_grade}")
print(f"                       (expected {EXPECTED_BY_GRADE})")
assert len(df) == EXPECTED_N, f"row count differs: {len(df):,}"
assert actual_by_grade == EXPECTED_BY_GRADE, \
    f"school level counts differ: {actual_by_grade}"
print(">>> counts match: the sample is unchanged, only the definition")

print()
print("=== Outcome under each definition ===")
print(f"  IDF (primary)      : {int(df['MS'].eq(1).sum()):,}  "
      f"({df['MS'].mean()*100:.2f}%)")
print(f"  ATP III (sensitivity): {int(df['MS_atp3'].eq(1).sum()):,}  "
      f"({df['MS_atp3'].mean()*100:.2f}%)")
print()
print("  These counts are before the missing-value and implausible-value")
print("  exclusions, so they are on 13,984 participants. The figures reported")
print("  in the abstract, the Results and Table 1 are the ones on the final")
print("  13,670, which notebook 03 produces.")


rows                 : 13,984  (expected 13,984)
by school level      : {1: 1586, 2: 5484, 3: 6914}
                       (expected {1: 1586, 2: 5484, 3: 6914})
>>> counts match: the sample is unchanged, only the definition

=== Outcome under each definition ===
  IDF (primary)      : 1,787  (12.78%)
  ATP III (sensitivity): 2,271  (16.24%)

  These counts are before the missing-value and implausible-value
  exclusions, so they are on 13,984 participants. The figures reported
  in the abstract, the Results and Table 1 are the ones on the final
  13,670, which notebook 03 produces.


In [26]:
# Save the analysis dataset after removing intermediate derivation fields.
df_final = df.rename(columns={'BMI_calc': 'bmi'}).drop(
    columns=['L', 'M', 'S', 'months', 'adult_wc_cut', 'wc_p90', 'age_floor',
             'EI_meat', 'EI_dairy', 'EI_fruit', 'EI_vegetable', 'EI_breakfast',
             'EI_ramen', 'EI_drink', 'EI_fastfood',
             'ramen', 'drink', 'fastfood', 'meat', 'dairy', 'fruit',
             'vegetable', 'breakfast'])

df_final.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')

flow = pd.DataFrame(FLOW)
flow.to_csv(OUT_FLOW, index=False, encoding='utf-8-sig')

print(f"saved -> {OUT_CSV}   ({len(df_final):,} rows, {df_final.shape[1]} cols)")
print(f"saved -> {OUT_FLOW}")
print()
print(flow.to_string(index=False))


saved -> D:\eda\mets\data\work\Obesity_Data_proprecessing.csv   (13,984 rows, 47 cols)
saved -> D:\eda\mets\output\tables\flow_01.csv

 step                                                stage  excluded  remaining
    0 Merged Student Health Examination records, 2021-2024         0     366903
    1                             Missing height or weight         1     366902
    2        Invalid date, grade-age mismatch, or age <= 5       165     366737
    3                                       Age < 10 years     85064     281673
    4   MetS components not measured (no biochemical exam)    266674      14999
    5      Weight-control items: conflicting or unanswered      1009      13990
    6                     Dietary items invalid or missing         6      13984
    7    Grade other than elementary 4 / middle 1 / high 1         0      13984
